<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Notebook 07</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">One front desk: every function is a skill</h1>
<p style="margin:.55em 0">Notebook 06 could only register. A real front desk also <b>verifies</b> callers, finds <b>free slots</b>,
<b>books</b>, <b>lists</b> and <b>cancels</b> appointments. Built as separate bots, they would each need their own
codes, their own idea of "who is calling", their own prompt. Built as <b>one system</b>:</p>
<div style="display:flex;flex-direction:column;align-items:flex-start;gap:4px;margin:.8em 0">
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>caller</b> ⇄ realtime model (one conversation, one prompt built from the skills)</div>
  <div style="opacity:.6;padding-left:18px">▼ proposes a call</div>
  <div style="border:2px solid #6366f1;background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>Session</b> (our code): who is verified · the one pending code · shared-phone choice<br>
    <span style="opacity:.7;font-size:.9em">guards run here: a <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">needs_patient</code> skill never runs for an unverified caller</span></div>
  <div style="opacity:.6;padding-left:18px">▼</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>skills</b>: verify_patient · confirm_code · choose_patient · start_registration · list_doctors ·
    find_slots · book_appointment · my_appointments · cancel_appointment</div>
  <div style="opacity:.6;padding-left:18px">▼</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>clinic</b>: one SQLite test DB, one SMS outbox, one OTP (live/clinic.py)</div>
</div>
<p style="margin:.55em 0">Adding a function is <b>one decorated Python function</b>: its description, parameters, whether it needs a
verified patient, and one line telling the model when to use it.</p>
<div style="margin:.8em 0;padding:.4em 1em;border-left:4px solid #6366f1;background:rgba(99,102,241,.12);border-radius:0 6px 6px 0">Rules come from DocAssist's appointment code (slots, leave, booking, cancelling); nothing touches DocAssist.</div>

</div>

In [1]:
import json
from datetime import datetime

from live import appointments, audio, clinic, frontdesk, realtime, tools

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">1. The clinic</h2>
<p style="margin:.55em 0">One test database for everything. Three doctors with weekly hours, three patients (Asha and her son Rohan share a
phone), and a few appointments placed relative to today, including a day of leave for Dr Kapoor.</p>

</div>

In [2]:
db = clinic.init_db(reset=True)
tue, wed, thu = (clinic.next_weekday(d) for d in ("Tuesday", "Wednesday", "Thursday"))
tue, wed, thu

(datetime.date(2026, 11, 24),
 datetime.date(2026, 11, 25),
 datetime.date(2026, 11, 26))

In [3]:
appointments.doctors(db)

[{'id': 1,
  'name': 'Dr. Anil Kapoor',
  'specialty': 'General Medicine',
  'days': ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']},
 {'id': 2,
  'name': 'Dr. Sunita Rao',
  'specialty': 'Pediatrics',
  'days': ['Monday', 'Wednesday', 'Friday']},
 {'id': 3,
  'name': 'Dr. Vikas Shah',
  'specialty': 'Cardiology',
  'days': ['Tuesday', 'Thursday']}]

In [4]:
def q(sql, *args):
    with clinic.db(db) as c:
        return [tuple(r) for r in c.execute(sql, args)]

q("SELECT Patient_ID, First_Name, Last_Name, age, Phone_Number FROM Patient")

[(1, 'asha', 'sharma', 41, '919812345678'),
 (2, 'rohan', 'sharma', 11, '919812345678'),
 (3, 'vikram', 'nair', 46, '919845012367')]

In [5]:
q("SELECT Appointment_ID, patient_id, doctor_id, appointment_date, status, coalesce(reason, notes) FROM Appointment")

[(1, 1, 1, '2026-11-24 10:00:00', 'scheduled', 'fever and cough'),
 (2, 2, 2, '2026-11-25 10:20:00', 'scheduled', 'vaccination'),
 (3, 3, 1, '2026-11-24 09:00:00', 'cancelled', 'back pain'),
 (4, None, 1, '2026-11-25 00:00:00', 'unavailable', 'full-day leave')]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">2. DocAssist's appointment rules, as functions</h2>
<p style="margin:.55em 0">Free slots come from the doctor's weekly hours, minus any non-cancelled appointment and any leave.
Asha holds Dr Kapoor's 10:00 on Tuesday:</p>

</div>

In [6]:
s = appointments.find_slots(db, "Dr Kapoor", f"{tue}")
s["free_count"], s["free"][:6]

(15, ['9:00 AM', '9:15 AM', '9:30 AM', '9:45 AM', '10:15 AM', '10:30 AM'])

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">A closed day says why, so the model can tell the caller:</p>

</div>

In [7]:
appointments.find_slots(db, "Kapoor", f"{wed}")["problem"], appointments.find_slots(db, "cardiology", f"{wed}")["problem"]

('Dr. Anil Kapoor is on leave that day',
 'Dr. Vikas Shah does not work on Wednesdays (works Tuesday, Thursday)')

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Doctors are found the way people say them, but only ones that exist:</p>

</div>

In [8]:
[appointments.find_slots(db, d, f"{tue}").get("doctor") or appointments.find_slots(db, d, f"{tue}")["status"]
 for d in ("the pediatrician", "Dr. Shah", "Dr. Smith")]

['Dr. Sunita Rao', 'Dr. Vikas Shah', 'unknown_doctor']

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">3. Skills: one registry, one decorator</h2>
<p style="margin:.55em 0">Every function the model can call is a <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">@skill</code> in <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">live/frontdesk.py</code>. The flag
<code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">needs_patient</code> is the guard:</p>

</div>

In [9]:
[(name, sk.needs_patient, list(sk.params)) for name, sk in frontdesk.SKILLS.items()]

[('verify_patient', False, ['phone']),
 ('confirm_code', False, ['code']),
 ('choose_patient', False, ['name']),
 ('start_registration',
  False,
  ['first_name', 'last_name', 'phone', 'sex', 'dob', 'age', 'email']),
 ('list_doctors', False, []),
 ('find_slots', False, ['doctor', 'date']),
 ('book_appointment', True, ['doctor', 'date', 'time', 'reason']),
 ('my_appointments', True, []),
 ('cancel_appointment', True, ['appointment_id'])]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Adding a function is one block. A caller asking "when are you open?" needs no patient, so no guard:</p>

</div>

In [10]:
@frontdesk.skill("The clinic's opening hours.", guide="when asked when the clinic is open.")
def clinic_hours(s):
    return {"hours": "Monday to Friday 9 AM to 5 PM; closed on weekends"}

list(frontdesk.SKILLS)

['verify_patient',
 'confirm_code',
 'choose_patient',
 'start_registration',
 'list_doctors',
 'find_slots',
 'book_appointment',
 'my_appointments',
 'cancel_appointment',
 'clinic_hours']

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">4. The session: state and guards, without the model</h2>
<p style="margin:.55em 0">A <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">Session</code> is one conversation. Its registry is what the model gets; <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">call</code> goes
through <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">tools.run</code>, the exact path a model's proposal takes (JSON args, validation, guard, skill).</p>

</div>

In [11]:
s = frontdesk.Session(db)
reg = s.registry()

def call(tool, **args):
    return tools.run(reg, tool, json.dumps(args))["result"]

call("my_appointments")

{'status': 'needs_verification',
 'problem': 'the caller is not verified yet',
 'next': 'Ask for their mobile number and call verify_patient.'}

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Unverified, so the guard answered, not the skill. Verify Asha's phone:</p>

</div>

In [12]:
call("verify_patient", phone="98123 45678")

{'status': 'code_sent',
 'sent_to': 'phone ending 5678',
 'next': 'Ask for the 6-digit code that was just texted.'}

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">No names yet: who is on a phone is only revealed after the code proves the caller holds it. The code went to the phone:</p>

</div>

In [13]:
s.otp.outbox[-1], s.state()

({'to': '+919812345678',
  'text': 'Your City Care Clinic verification code is 883636.'},
 {'patient': None,
  'code_pending_for': 'verification',
  'choosing_between': None})

In [14]:
call("confirm_code", code="000000")

{'status': 'wrong_code', 'tries_left': 2}

In [15]:
call("confirm_code", code=s.otp.last_code())

{'status': 'choose_patient',
 'names': ['Asha Sharma', 'Rohan Sharma'],
 'next': 'Several patients share this phone: ask which of them they are calling about.'}

In [16]:
call("my_appointments")   # phone proven, but which person?

{'status': 'needs_verification',
 'problem': 'the caller is not verified yet',
 'next': 'Ask for their mobile number and call verify_patient.'}

In [17]:
call("choose_patient", name="Asha"), s.state()

({'status': 'verified',
  'name': 'Asha Sharma',
  'next': 'Greet Asha, then carry on with what they already asked for (for example call my_appointments if they wanted their appointments). Ask only if they asked nothing yet.'},
 {'patient': {'id': 1, 'name': 'Asha Sharma'},
  'code_pending_for': None,
  'choosing_between': None})

In [18]:
call("my_appointments")

{'patient': 'Asha Sharma',
 'appointments': [{'appointment_id': 1,
   'doctor': 'Dr. Anil Kapoor',
   'specialty': 'General Medicine',
   'date': 'Tuesday 24 November 2026',
   'time': '10:00 AM',
   'reason': 'fever and cough'}]}

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Even on the same phone, Asha cannot touch Rohan's appointment (id 2): it looks exactly like a missing one.</p>

</div>

In [19]:
call("cancel_appointment", appointment_id=2)

{'status': 'not_found', 'problem': 'no appointment 2 for this patient'}

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">5. One prompt, built from the skills</h2>
<p style="margin:.55em 0">The shared rules (verify first, dates as YYYY-MM-DD, confirm before writing) plus one line per skill.
Today's date is in it, so "next Thursday" can become a date.</p>

</div>

In [20]:
print(s.instructions())

You are the voice front desk of City Care Clinic. Today is Monday 23 November 2026.
Be brief and warm. Ask one thing at a time.
Use the tools for every fact: never invent doctors, times, codes, appointments or patient details.
When a tool result has a "next", do what it says.
When you have what a tool needs, call it in that same turn. Never announce an action ("I'll send a code", "I'll book that") instead of calling the tool.
Anything about an existing patient (their appointments, booking, cancelling) needs a verified caller: ask for their mobile number, call verify_patient, then confirm_code with the code they read from the SMS. A new patient becomes verified by registering.
The moment the caller gives a mobile number, call verify_patient in that same turn. Codes are sent only by verify_patient or start_registration: never say a code was or will be sent unless one returned code_sent.
Copy phone numbers and codes digit for digit as the caller said them.
Give tools dates as YYYY-MM-DD, 

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">6. In text first: verify on a shared phone, then appointments</h2>
<p style="margin:.55em 0"><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">say_text</code> runs tool calls until the model answers in words. Each tool call prints its result.</p>
<p style="margin:.55em 0">OpenAI sometimes lets one connection go silent mid-reply (measured: about 1 long reply in 6, server-side).
Because who is verified and which code is pending live in <b>our</b> Session, a new connection only needs the
words so far: <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">realtime.replay</code> sends them, plus our state, and the conversation carries on.</p>

</div>

In [21]:
def text_session():
    global s, ws, history
    s, ws, history = frontdesk.Session(db), None, []

async def connect():
    global ws
    ws = await realtime.start_session(s.instructions(), tools=[sp for sp, _ in s.registry().values()],
                                      max_output_tokens=1000)
    if history:  # after a drop: the words and tool calls so far; the Session kept the state
        await realtime.replay(ws, history, f"Our state: {json.dumps(s.state())}")

async def say_text(text, resumes=2):
    reg = s.registry()
    if ws is None:
        await connect()
    history.append(("user", text))
    await realtime.send(ws, realtime.user_text(text))
    for attempt in range(resumes + 1):
        try:
            await realtime.send(ws, {"type": "response.create"})
            while True:
                ev = await realtime.collect(ws, until="response.done", timeout=15)
                handled = await realtime.answer_tool_calls(ws, ev[-1]["response"], reg)
                for name, args, out in handled:
                    print("TOOL", name, " ".join(args.split()), "->", json.dumps(out))
                    history.append(("system", f"Already done: {name}({' '.join(args.split())}) -> {json.dumps(out)}"))
                if not handled:
                    said = [e["transcript"] for e in ev if e["type"] == "response.output_audio_transcript.done"]
                    history.append(("assistant", " ".join(said)))
                    return said
        except realtime.DROPPED as e:
            if attempt == resumes:
                raise
            print(f"RESUME after {type(e).__name__}: new connection, replaying {len(history)} items")
            await connect()  # replays everything, including this turn and the tools it already ran

text_session()

In [22]:
await say_text("Hi, I would like to check my appointments. My mobile is 9812345678.")

TOOL verify_patient { "phone": "9812345678" } -> {"result": {"status": "code_sent", "sent_to": "phone ending 5678", "next": "Ask for the 6-digit code that was just texted."}}


['Please tell me the 6-digit code you received via text.']

In [23]:
await say_text(f"The code is {s.otp.last_code()}.")

TOOL confirm_code { "code": "163081" } -> {"result": {"status": "choose_patient", "names": ["Asha Sharma", "Rohan Sharma"], "next": "Several patients share this phone: ask which of them they are calling about."}}


['Which patient are you calling about—Asha Sharma or Rohan Sharma?']

In [24]:
await say_text("I am calling for Asha.")

TOOL choose_patient { "name": "Asha Sharma" } -> {"result": {"status": "verified", "name": "Asha Sharma", "next": "Greet Asha, then carry on with what they already asked for (for example call my_appointments if they wanted their appointments). Ask only if they asked nothing yet."}}


TOOL my_appointments {} -> {"result": {"patient": "Asha Sharma", "appointments": [{"appointment_id": 1, "doctor": "Dr. Anil Kapoor", "specialty": "General Medicine", "date": "Tuesday 24 November 2026", "time": "10:00 AM", "reason": "fever and cough"}]}}


['Here’s the upcoming appointment for Asha Sharma: with Dr. Anil Kapoor, in General Medicine, on Tuesday 24 November 2026, at 10:00 AM for fever and cough. \n\nDoes that look correct?']

In [25]:
await ws.close()
s.state()

{'patient': {'id': 1, 'name': 'Asha Sharma'},
 'code_pending_for': None,
 'choosing_between': None}

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">7. In text: a new patient registers and books, in one conversation</h2>
<p style="margin:.55em 0">Registering verifies the caller, so the same session can book straight away.</p>

</div>

In [26]:
text_session()
await say_text("Hello, I am new here. Meera Iyer, female, born 3rd April 1995, mobile 9444123456.")

TOOL start_registration { "first_name": "Meera", "last_name": "Iyer", "phone": "9444123456", "sex": "Female", "dob": "1995-04-03", "age": "", "email": "" } -> {"result": {"status": "code_sent", "sent_to": "phone ending 3456", "read_back": {"name": "Meera Iyer", "spelled": "M-E-E-R-A  I-Y-E-R", "phone_digits": "9 4 4 4 1 2 3 4 5 6", "sex": "Female", "date_of_birth": "03 April 1995", "age": 31}, "next": "First read back every detail in read_back: spell both names letter by letter and say the phone digits one by one. Then ask for the 6-digit code that was just texted."}}


['Thank you, let me confirm your details. Your name is Meera Iyer, spelled M-E-E-R-A I-Y-E-R. Your mobile number is 9444 123456, and your date of birth is 3rd April 1995.\n\nNow, could you please provide the 6-digit code that was sent to your phone?']

In [27]:
await say_text(f"Yes, all correct. The code is {s.otp.last_code()}.")

TOOL confirm_code { "code": "494645" } -> {"result": {"status": "registered", "patient_id": 4, "name": "Meera Iyer", "next": "Give them their patient ID, then ask if they would like to book an appointment."}}


["You're all set, Meera. Your patient ID is 4. Would you like to book an appointment now? If so, let me know the doctor or the reason for the visit."]

In [28]:
await say_text(f"Please book me with the cardiologist on {thu:%A} {thu.day} {thu:%B} at 3 PM for a chest pain check.")

TOOL find_slots { "doctor": "cardiologist", "date": "2026-11-26" } -> {"result": {"status": "ok", "doctor": "Dr. Vikas Shah", "specialty": "Cardiology", "date": "Thursday 26 November 2026", "free": ["2:00 PM", "2:30 PM", "3:00 PM", "3:30 PM", "4:00 PM", "4:30 PM"], "free_count": 6, "next": "If the caller already asked for a time that is in free, confirm that one; otherwise offer two or three free times. Never read the whole list."}}


['The 3 PM slot is available. Shall I go ahead and book you with Dr. Vikas Shah for Thursday, 26 November at 3 PM?']

In [29]:
await say_text("Yes, please book it.")

TOOL book_appointment { "doctor": "Dr. Vikas Shah", "date": "2026-11-26", "time": "3:00 PM", "reason": "chest pain check" } -> {"result": {"status": "booked", "appointment_id": 5, "doctor": "Dr. Vikas Shah", "date": "Thursday 26 November 2026", "time": "3:00 PM", "reason": "chest pain check"}}


['Your appointment is booked with Dr. Vikas Shah on Thursday, 26 November 2026 at 3:00 PM for a chest pain check. If you need anything else, let me know.']

In [30]:
await ws.close()
q("SELECT a.Appointment_ID, p.First_Name, a.doctor_id, a.appointment_date, a.status, a.reason "
  "FROM Appointment a JOIN Patient p ON p.Patient_ID = a.patient_id WHERE p.First_Name = 'meera'")

[(5, 'meera', 3, '2026-11-26 15:00:00', 'scheduled', 'chest pain check')]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">8. By voice</h2>
<p style="margin:.55em 0">A <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">ScriptedUser</code> speaks each line after the model's answer has played. Code lines are functions:
they read the phone's latest SMS at the moment they are spoken.</p>

</div>

In [31]:
spk = audio.Speaker(); spk.stream.start()

def code_line(prefix=""):
    return lambda: f"{prefix}The code is {', '.join(s.otp.last_code())}."

async def voice_run(lines, seconds=90):
    global s
    s = frontdesk.Session(db)  # a fresh conversation
    user = audio.ScriptedUser(lines, spk)
    reg = s.registry()
    start = lambda: realtime.start_session(s.instructions(), tools=[sp for sp, _ in reg.values()],
                                           turn_detection=frontdesk.VAD, max_output_tokens=1000)
    # resilient: a stalled connection is replaced, the words so far + our Session state are replayed
    return await realtime.converse_resilient(start, user.chunks(), spk, seconds=seconds, tools=reg,
                                             on_note=user.on_note, stop=user.done,
                                             context=lambda: f"Our state: {json.dumps(s.state())}")

def story(log):
    return [row for row in log if row[1] in ("user_said", "tool_call", "model_said", "RESUME")]

VIKRAM = "9 8 4 5 0 1 2 3 6 7"

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h3 style="font-size:1.05em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">8a. Verify, then check appointments</h3><p style="margin:.55em 0">Vikram's only appointment was cancelled, so he has none upcoming.</p>

</div>

In [32]:
log = await voice_run([
    f"Hi, this is Vikram. I want to check my appointments. My mobile is {VIKRAM}.",
    code_line(),
])
story(log)

0.32 input_audio_buffer.speech_started 0.00s of audio still queued


9.49 input_audio_buffer.speech_stopped 9600
9.49 input_audio_buffer.committed 
9.5 response.created in_progress


9.75 filler "Let me check." while verify_patient runs


9.98 response.done completed
9.98 tool_call verify_patient({ "phone": "9845012367" }) -> {"result": {"status": "code_sent", "sent_to": "phone ending 2367", "next": "Ask for the 6-digit code that was just texted."}}
10.07 user_said Hi, this is Vikram. I want to check my appointments. My mobile is 9845012367.


10.38 response.created in_progress


10.69 first_audio_delta 


... (8 outputs left out)

27.75 response.created in_progress


28.07 first_audio_delta 


28.73 model_said It looks like you don’t have any upcoming appointments at the moment. Would you like to book one? If so, could you tell me the doctor, the day, and the time you're looking for?
28.74 response.done completed


[(9.98,
  'tool_call',
  'verify_patient({ "phone": "9845012367" }) -> {"result": {"status": "code_sent", "sent_to": "phone ending 2367", "next": "Ask for the 6-digit code that was just texted."}}'),
 (10.07,
  'user_said',
  'Hi, this is Vikram. I want to check my appointments. My mobile is 9845012367.'),
 (11.21,
  'model_said',
  'I’ve sent a 6-digit code to your phone ending in 2367. Could you please read it out to me?'),
 (25.75,
  'tool_call',
  'confirm_code({ "code": "066927" }) -> {"result": {"status": "verified", "name": "Vikram Nair", "next": "Greet Vikram, then carry on with what they already asked for (for example call my_appointments if they wanted their appointments). Ask only if they asked nothing yet."}}'),
 (25.75, 'user_said', 'The code is 066927.'),
 (27.12,
  'model_said',
  'Hello Vikram! Thanks for verifying. Now, let me pull up your appointments. One moment.'),
 (27.32,
  'tool_call',
  'my_appointments({}) -> {"result": {"patient": "Vikram Nair", "appointments"

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h3 style="font-size:1.05em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">8b. Verify, then book a slot</h3>

</div>

In [33]:
log = await voice_run([
    f"Hi, I would like to book an appointment with Doctor Kapoor on {thu:%A} {thu.day} {thu:%B} at 10 AM "
    f"for back pain. My mobile is {VIKRAM}.",
    code_line(),
    "Yes, please book it.",
])
story(log)

0.33 input_audio_buffer.speech_started 0.00s of audio still queued


12.71 input_audio_buffer.speech_stopped 12352
12.71 input_audio_buffer.committed 
12.71 response.created in_progress


12.98 filler "Let me check." while verify_patient runs
13.15 response.done completed
13.15 tool_call verify_patient({ "phone": "9845012367" }) -> {"result": {"status": "code_sent", "sent_to": "phone ending 2367", "next": "Ask for the 6-digit code that was just texted."}}


13.31 user_said Hi, I would like to book an appointment with Dr. Kapoor on Thursday November 26th at 10 a.m. for back pain. My mobile is 9845012367.


13.57 response.created in_progress


13.95 first_audio_delta 


... (12 outputs left out)

72.67 response.created in_progress


72.97 first_audio_delta 


73.88 model_said Your appointment with Dr. Anil Kapoor is booked for Thursday, November 26, 2026, at 10:00 AM for back pain. If you need anything else, let me know.
73.89 response.done completed


[(13.15,
  'tool_call',
  'verify_patient({ "phone": "9845012367" }) -> {"result": {"status": "code_sent", "sent_to": "phone ending 2367", "next": "Ask for the 6-digit code that was just texted."}}'),
 (13.31,
  'user_said',
  'Hi, I would like to book an appointment with Dr. Kapoor on Thursday November 26th at 10 a.m. for back pain. My mobile is 9845012367.'),
 (14.32,
  'model_said',
  'Please provide the 6-digit code that was sent to your phone.'),
 (54.64,
  'tool_call',
  'confirm_code({ "code": "787838" }) -> {"result": {"status": "verified", "name": "Vikram Nair", "next": "Greet Vikram, then carry on with what they already asked for (for example call my_appointments if they wanted their appointments). Ask only if they asked nothing yet."}}'),
 (54.64, 'user_said', 'The code is 7873838.'),
 (57.1,
  'model_said',
  "Hello Vikram, great to have you verified. Now, to confirm: you'd like to book an appointment with Dr. Kapoor on Thursday, November 26th at 10 AM for back pain. Shall 

In [34]:
q("SELECT Appointment_ID, doctor_id, appointment_date, status, reason FROM Appointment WHERE patient_id = 3")

[(3, 1, '2026-11-24 09:00:00', 'cancelled', 'back pain'),
 (6, 1, '2026-11-26 10:00:00', 'scheduled', 'back pain')]

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h3 style="font-size:1.05em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">8c. Booking before verifying</h3>
<p style="margin:.55em 0">Whatever the model does, the guard is in code: no appointment can appear for an unverified caller.</p>

</div>

In [35]:
before = q("SELECT COUNT(*) FROM Appointment")
log = await voice_run([
    "Book me with Doctor Kapoor tomorrow at 11 AM for a fever, please.",
    "11 AM is fine, please book it.",
])
story(log), before == q("SELECT COUNT(*) FROM Appointment"), s.state()

0.43 input_audio_buffer.speech_started 0.00s of audio still queued


4.55 input_audio_buffer.speech_stopped 5504
4.55 input_audio_buffer.committed 
4.56 response.created in_progress


5.27 first_audio_delta 


6.91 user_said Book me with Dr. Kapoor tomorrow at 11 a.m. for a fever, please.
7.07 model_said Sure. I'll find out the available slots for Dr. Kapoor tomorrow and confirm the exact time. Let me check that now.
7.07 filler "Let me check." while find_slots runs
7.07 response.done completed
7.08 tool_call find_slots({ "doctor": "Dr. Kapoor", "date": "2026-11-24" }) -> {"result": {"status": "ok", "doctor": "Dr. Anil Kapoor", "specialty": "General Medicine", "date": "Tuesday 24 November 2026", "free": ["9:00 AM", "9:15 AM", "9:30 AM", "9:45 AM", "10:15 AM", "10:30 AM", "10:45 AM", "11:00 AM", "11:15 AM", "11:30 AM", "11:45 AM", "12:00 PM", "12:15 PM", "12:30 PM", "12:45 PM"], "free_count": 15, "next": "If the caller already asked for a time that is in free, confirm that one; otherwise offer two or three free times. Never read the whole list."}}


7.9 response.created in_progress


8.64 first_audio_delta 
8.79 model_said I can book Dr. Anil Kapoor for tomorrow at 11 AM. Does that time work for you?
8.79 response.done completed


20.15 input_audio_buffer.speech_started 0.00s of audio still queued


24.08 input_audio_buffer.speech_stopped 22976
24.09 input_audio_buffer.committed 
24.09 response.created in_progress


24.62 user_said 11 a.m. is fine, please book it.
24.75 first_audio_delta 


25.23 model_said Great. Could you please provide your mobile number so I can verify your details before booking?
25.24 response.done completed


([(6.91,
   'user_said',
   'Book me with Dr. Kapoor tomorrow at 11 a.m. for a fever, please.'),
  (7.07,
   'model_said',
   "Sure. I'll find out the available slots for Dr. Kapoor tomorrow and confirm the exact time. Let me check that now."),
  (7.08,
   'tool_call',
   'find_slots({ "doctor": "Dr. Kapoor", "date": "2026-11-24" }) -> {"result": {"status": "ok", "doctor": "Dr. Anil Kapoor", "specialty": "General Medicine", "date": "Tuesday 24 November 2026", "free": ["9:00 AM", "9:15 AM", "9:30 AM", "9:45 AM", "10:15 AM", "10:30 AM", "10:45 AM", "11:00 AM", "11:15 AM", "11:30 AM", "11:45 AM", "12:00 PM", "12:15 PM", "12:30 PM", "12:45 PM"], "free_count": 15, "next": "If the caller already asked for a time that is in free, confirm that one; otherwise offer two or three free times. Never read the whole list."}}'),
  (8.79,
   'model_said',
   'I can book Dr. Anil Kapoor for tomorrow at 11 AM. Does that time work for you?'),
  (24.62, 'user_said', '11 a.m. is fine, please book it.'),
  (

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h3 style="font-size:1.05em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">8d. A wrong code, then the right one</h3>

</div>

In [36]:
def wrong_code_line():
    real = s.otp.last_code()
    wrong = "".join(str((int(ch) + 1) % 10) for ch in real)  # every digit off by one
    return f"The code is {', '.join(wrong)}."

log = await voice_run([
    f"Hi, I want to check my appointments. My mobile is {VIKRAM}.",
    wrong_code_line,
    code_line("Sorry, I read it wrong. "),
])
story(log)

0.44 input_audio_buffer.speech_started 0.00s of audio still queued


7.52 input_audio_buffer.speech_stopped 7840
7.52 input_audio_buffer.committed 
7.54 response.created in_progress


7.86 filler "Let me check." while verify_patient runs
7.98 response.done completed
7.98 tool_call verify_patient({ "phone": "9845012367" }) -> {"result": {"status": "code_sent", "sent_to": "phone ending 2367", "next": "Ask for the 6-digit code that was just texted."}}
7.98 user_said Hi, I want to check my appointments. My mobile is 9845012367.


8.48 response.created in_progress


8.83 first_audio_delta 


9.91 model_said Please provide the 6-digit code that was sent to your phone.
9.91 response.done completed


... (14 outputs left out)

45.94 response.created in_progress


46.31 first_audio_delta 


47.34 model_said Here’s your upcoming appointment:

With Dr. Anil Kapoor, in General Medicine, on Thursday, 26 November 2026, at 10:00 AM for back pain.

Does that sound correct?
47.34 response.done completed


[(7.98,
  'tool_call',
  'verify_patient({ "phone": "9845012367" }) -> {"result": {"status": "code_sent", "sent_to": "phone ending 2367", "next": "Ask for the 6-digit code that was just texted."}}'),
 (7.98,
  'user_said',
  'Hi, I want to check my appointments. My mobile is 9845012367.'),
 (9.91,
  'model_said',
  'Please provide the 6-digit code that was sent to your phone.'),
 (23.31,
  'tool_call',
  'confirm_code({ "code": "300448" }) -> {"result": {"status": "wrong_code", "tries_left": 2}}'),
 (23.31, 'user_said', 'The code is three, zero, zero, four, four, eight.'),
 (25.04,
  'model_said',
  'It seems the code you provided isn’t correct. You have two tries left. Could you double-check the code on your phone and let me know the exact numbers?'),
 (43.47, 'user_said', 'Sorry, I read it wrong. The code is 299337.'),
 (43.48,
  'tool_call',
  'confirm_code({ "code": "299337" }) -> {"result": {"status": "verified", "name": "Vikram Nair", "next": "Greet Vikram, then carry on with wha

In [37]:
spk.stream.close()

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Takeaways</span>
<ul style="margin:.4em 0;padding-left:1.4em">
  <li style="margin:.2em 0"><b>One system:</b> one session per conversation holds identity, the one pending code and the shared-phone choice.
      Registration and verification share <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">confirm_code</code>.</li>
  <li style="margin:.2em 0"><b>Scaling = one decorator:</b> description, parameters, <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">needs_patient</code>, and a one-line guide.
      The prompt and the registry follow automatically.</li>
  <li style="margin:.2em 0"><b>Guards in code:</b> unverified callers cannot read or change appointments, and one patient cannot touch
      another's appointment, whatever the model proposes.</li>
  <li style="margin:.2em 0"><b>Numbers are fragile:</b> in a first run the model miscopied a number with long repeats (99000 11122 became 9 or 11 digits) although the transcript was right. Validation caught it; a real desk should read numbers back.</li> names on a phone are revealed only after the code proves the caller holds the phone.</li>
</ul>
<p style="margin:.55em 0"><b>Next:</b> the web app runs this same <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">frontdesk.Session</code>.</p>

</div>